# Experiment Run Guide

Этот ноутбук собирает команды для экспериментов, которые добавлены в последних коммитах:

- Lorenz baseline experiments в `lorenz_baseline/`.
- Synthetic sea-ice ensemble evaluation в `synthetic_eval/`.
- Conditional kNN vs concat notebook `experiments_conditional_knn_concat_tests.ipynb`.

По умолчанию тяжелые команды не запускаются. Поставь нужный `RUN_* = True` в соответствующей ячейке.

In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

# Server defaults. Override env vars if server layout changes.
REPO_DIR = Path(os.environ.get("REPO_DIR", "/home"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/mnt/sciml/a.sadreev/sea_ice_data"))
PYTHON = os.environ.get("PYTHON", "python")

os.chdir(REPO_DIR)
print("repo_dir =", Path.cwd())
print("data_root =", DATA_ROOT)
print("python =", PYTHON)

def run(cmd, check=True):
    print("$", " ".join(shlex.quote(str(x)) for x in cmd))
    return subprocess.run([str(x) for x in cmd], check=check)

## 1. Fast Synthetic Smoke Test

Проверяет pipeline без настоящей concat-модели через `dummy` runner. Берет первые 2 поля из `valid`, генерирует random/block observations и считает метрики. Это быстрый sanity check после pull на сервере.

In [ ]:
RUN_SYNTHETIC_SMOKE = False

if RUN_SYNTHETIC_SMOKE:
    run([
        PYTHON, "-m", "synthetic_eval.cli",
        "--input-dir", DATA_ROOT / "valid",
        "--output-dir", DATA_ROOT / "synthetic_eval_smoke",
        "--runner", "dummy",
        "--ensemble-size", "4",
        "--mask-types", "random,block",
        "--densities", "0.05",
        "--noise-levels", "0.0",
        "--max-timesteps", "2",
        "--rank-stride", "16",
        "--energy-score-limit", "1000",
        "--no-plots",
    ])

## 2. Synthetic Eval: First 30 Validation Samples x 15

Это настройка, которую мы обсудили последней: первые 30 файлов из `valid`, 15 ensemble members на каждое поле. Итого `30 * 15 = 450` generated analysis samples для каждой комбинации mask/density/noise.

Если `--concat-run-dir` не указан, скрипт ищет последний `ema_best_model.pth` под `checkpoints/**` относительно `REPO_DIR`.

In [ ]:
RUN_SYNTHETIC_FIRST30_CONCAT = False

if RUN_SYNTHETIC_FIRST30_CONCAT:
    run([
        PYTHON, "-m", "synthetic_eval.cli",
        "--input-dir", DATA_ROOT / "valid",
        "--output-dir", DATA_ROOT / "synthetic_eval_val_first30x15",
        "--runner", "concat",
        "--checkpoint-name", "ema_best_model.pth",
        "--n-cases", "30",
        "--case-selection", "first",
        "--ensemble-size", "15",
        "--mask-types", "random,block",
        "--densities", "0.05",
        "--noise-levels", "0.0",
        "--num-timesteps", "50",
        "--method", "euler",
    ])

## 3. Synthetic Eval: Daily 365 x 15

Для годового почасового датасета берет один `x_true` каждые 24 файла: индексы `0, 24, 48, ..., 8736`. Итого 365 ground-truth cases и 15 ensemble members на каждое поле.

In [ ]:
RUN_SYNTHETIC_DAILY365_CONCAT = False

if RUN_SYNTHETIC_DAILY365_CONCAT:
    run([
        PYTHON, "-m", "synthetic_eval.cli",
        "--input-dir", DATA_ROOT / "valid",
        "--output-dir", DATA_ROOT / "synthetic_eval_daily_365x15",
        "--runner", "concat",
        "--checkpoint-name", "ema_best_model.pth",
        "--n-cases", "365",
        "--case-selection", "daily",
        "--daily-stride", "24",
        "--ensemble-size", "15",
        "--mask-types", "random,block",
        "--densities", "0.01,0.05,0.10,0.25",
        "--noise-levels", "0.0",
        "--num-timesteps", "50",
        "--method", "euler",
    ])

## 4. Outputs To Inspect

Synthetic eval пишет в `--output-dir`:

- `metadata.json` - параметры запуска и выбранные case indices.
- `per_case_metrics.csv` - per-case contributions для последующего bootstrap.
- `aggregate_metrics.csv/json` - средние метрики по `variable/mask_type/density/noise_level`.
- `spread_skill_bins.csv` - данные для spread-skill plot.
- `arrays/rank_histograms.npz` - counts rank histogram с `M + 1` bins.
- `plots/*.png` - rank histogram, coverage, spread-skill, RMSE/CRPS vs density, example panels.

In [ ]:
OUT_DIR = DATA_ROOT / "synthetic_eval_val_first30x15"
if OUT_DIR.exists():
    for path in sorted(OUT_DIR.glob("*")):
        print(path)
else:
    print("Output dir does not exist yet:", OUT_DIR)

## 5. Lorenz Baseline

Lorenz baseline содержит генератор датасета и два notebooks: concat и gradient methods. Датасет уже закоммичен в `lorenz_baseline/data`, но если нужно пересоздать его на сервере, используй команду ниже.

In [ ]:
RUN_LORENZ_DATASET_REGEN = False

if RUN_LORENZ_DATASET_REGEN:
    run([
        PYTHON, "-m", "lorenz_baseline.generate_dataset",
        "--output-dir", "lorenz_baseline/data",
        "--train-size", "200000",
        "--val-size", "20000",
        "--test-size", "20000",
        "--workers", "8",
    ])

Execute Lorenz notebooks from shell/Jupyter if needed. These can be heavy because they train models:

In [ ]:
RUN_LORENZ_CONCAT_NOTEBOOK = False
RUN_LORENZ_GRADIENT_NOTEBOOK = False

if RUN_LORENZ_CONCAT_NOTEBOOK:
    run([PYTHON, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute", "--inplace", "lorenz_baseline/concat_method.ipynb"])

if RUN_LORENZ_GRADIENT_NOTEBOOK:
    run([PYTHON, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute", "--inplace", "lorenz_baseline/gradient_method.ipynb"])

## 7. Batch kNN Baseline By Day

Этот CLI ищет ближайшие train-поля для каждого выбранного validation дня по sparse condition `y = x_ref * mask`. Он не генерирует concat samples; это baseline через поиск ближайших из датасета.

Outputs:

- `selected_references.csv` - какие validation days выбраны.
- `knn_neighbors.csv` - top-1 nearest neighbor для каждого дня с `condition_mse_norm` и full-field MSE.
- `summary.json` - параметры запуска.


In [ ]:
RUN_KNN_DECEMBER = False

if RUN_KNN_DECEMBER:
    run([
        PYTHON, "-m", "synthetic_eval.knn_baseline",
        "--data-root", DATA_ROOT,
        "--output-dir", DATA_ROOT / "knn_baseline_december",
        "--selection", "december",
        "--n-days", "31",
        "--k-neighbors", "1",
        "--mask-type", "swath",
        "--n-tracks-min", "7",
        "--n-tracks-max", "7",
        "--seed", "20260407",
    ])


### 30 Consecutive Days

Задай `--start-date`, и скрипт возьмет один первый validation файл на каждый день от этой даты, всего 30 дней. Если какой-то день отсутствует в split, он будет пропущен, а фактическое число дней будет видно в `summary.json`.

In [ ]:
RUN_KNN_30_CONSECUTIVE = False
KNN_START_DATE = "2022-12-01"

if RUN_KNN_30_CONSECUTIVE:
    run([
        PYTHON, "-m", "synthetic_eval.knn_baseline",
        "--data-root", DATA_ROOT,
        "--output-dir", DATA_ROOT / f"knn_baseline_30days_{KNN_START_DATE}",
        "--selection", "consecutive",
        "--start-date", KNN_START_DATE,
        "--n-days", "30",
        "--k-neighbors", "1",
        "--mask-type", "swath",
        "--n-tracks-min", "7",
        "--n-tracks-max", "7",
        "--seed", "20260407",
    ])


## 6. Conditional kNN vs Concat Notebook

`experiments_conditional_knn_concat_tests.ipynb` сравнивает локальные train kNN neighbors с concat samples при одном и том же sparse condition. Его лучше запускать как отдельный notebook, потому что он содержит визуализации и permutation tests.

Серверные дефолты внутри notebook уже такие:

- `REPO_DIR=/home`
- `DATA_ROOT=/mnt/sciml/a.sadreev/sea_ice_data`
- `STATS_JSON=$DATA_ROOT/train/stats.json`
- `MASK_PATH=$DATA_ROOT/mask_padding.npy`

In [ ]:
RUN_KNN_CONCAT_NOTEBOOK = False

if RUN_KNN_CONCAT_NOTEBOOK:
    run([PYTHON, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute", "--inplace", "experiments_conditional_knn_concat_tests.ipynb"])